# M1.3: NumPy and SciPy for energy numerics — and why power flow needs Newton

## Objectives

At the end of this notebook you should be able to:

- compute energy-system quantities over thousands of hours with vectorised NumPy and broadcasting
- solve large network equations efficiently with sparse matrices
- explain why constant-power loads make network equations **nonlinear**, and what voltage collapse is
- derive and implement **Newton's method** for one equation and for a system of equations (with a Jacobian)
- recognise quadratic convergence, and validate a nonlinear network solution with a power balance

This lesson also covers **Maths-1.2** of the foundations track. In EE-1.1 every source injected a fixed *current*, so the network equations were linear and one `np.linalg.solve` was enough. Real loads draw a fixed *power* — and because P = V·I, the equations become nonlinear. That single fact is why AC power flow in Module 04 needs Newton's method, and today you build it on a problem small enough to check by hand.

In [1]:
import time

import numpy as np
import pandas as pd
from scipy import optimize, sparse
from scipy.sparse.linalg import spsolve

### Vectorisation: thousands of hours at once

Energy models work with **arrays of hours × assets**: 8,760 hours times dozens of generators. Python loops over that are slow and easy to get wrong; NumPy operates on whole arrays at once.

Below, a **synthetic** availability matrix (8,760 hours × 6 generators, random but plausible) is combined with capacities using **broadcasting**: an array of shape (8760, 6) multiplied by one of shape (6,) applies each capacity to its own column.

In [2]:
rng = np.random.default_rng(0)
names = np.array(["wind", "solar", "hydro", "nuclear", "coal", "gas"])
p_nom_MW = np.array([1000, 800, 1500, 350, 210, 700])
availability = np.column_stack([rng.uniform(0.0, 0.9, 8760), np.clip(rng.normal(0.15, 0.2, 8760), 0, 1),
                                np.full(8760, 0.45), np.full(8760, 0.95), np.full(8760, 0.9), np.full(8760, 0.92)])

available_MW = availability * p_nom_MW            # (8760, 6) * (6,) -> (8760, 6)
energy_GWh = available_MW.sum(axis=0) / 1e3        # one value per generator
capacity_factor = available_MW.mean(axis=0) / p_nom_MW
pd.DataFrame({"available_energy_GWh": energy_GWh.round(0), "capacity_factor": capacity_factor.round(3)}, index=names)

         available_energy_GWh  capacity_factor
wind                   3930.0            0.449
solar                  1240.0            0.177
hydro                  5913.0            0.450
nuclear                2913.0            0.950
coal                   1656.0            0.900
gas                    5641.0            0.920

`axis=0` sums *down* the hours, leaving one number per generator. The capacity factor is just the mean availability — the broadcasting did the per-generator scaling for you. Now compare the speed with an explicit double loop:

In [3]:
t0 = time.perf_counter()
energy_loop = [sum(availability[t, g] * p_nom_MW[g] for t in range(8760)) for g in range(6)]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
energy_vec = (availability * p_nom_MW).sum(axis=0)
t_vec = time.perf_counter() - t0

assert np.allclose(energy_loop, energy_vec)
print(f"loop {t_loop * 1e3:.1f} ms, vectorised {t_vec * 1e3:.3f} ms  (~{t_loop / t_vec:.0f}x faster)")

loop 14.7 ms, vectorised 0.463 ms  (~32x faster)


Same numbers, a large speed-up — and the vectorised line is also easier to read. Exact timings depend on your machine; the ratio is what matters. Rule for this course: **if you write a loop over hours, stop and look for the array operation.**

### Sparse linear algebra: why grids are cheap to solve

In EE-1.1 you solved **G · V = I**. In a real grid each node connects to only a few others, so almost every entry of **G** is zero — the matrix is **sparse**. Storing and solving only the non-zeros is dramatically cheaper.

Here is a 3,000-node chain network (each node connected to its neighbours, 1 S per branch), with 1 A injected at the far end and node 0 as reference:

In [4]:
n_nodes = 3000
main = np.full(n_nodes, 2.0)
main[-1] = 1.0                                   # last node has only one neighbour
off = np.full(n_nodes - 1, -1.0)
G_sparse = sparse.diags([off, main, off], [-1, 0, 1], format="csc")
G_dense = G_sparse.toarray()
I = np.zeros(n_nodes)
I[-1] = 1.0

t0 = time.perf_counter(); V_dense = np.linalg.solve(G_dense, I); t_dense = time.perf_counter() - t0
t0 = time.perf_counter(); V_sparse = spsolve(G_sparse, I); t_sparse = time.perf_counter() - t0

assert np.allclose(V_dense, V_sparse)
assert np.isclose(V_sparse[-1], n_nodes), "1 A through 3,000 branches of 1 ohm each"
print(f"non-zeros: {G_sparse.nnz:,} of {n_nodes**2:,} entries ({100 * G_sparse.nnz / n_nodes**2:.2f} %)")
print(f"dense {t_dense:.3f} s, sparse {t_sparse:.4f} s")

non-zeros: 8,998 of 9,000,000 entries (0.10 %)
dense 0.397 s, sparse 0.0015 s


Only 0.1 % of the entries are non-zero, and the sparse solver is much faster with the same answer. For the European grid with tens of thousands of buses, dense matrices would not even fit in memory. PyPSA and pandapower use sparse matrices internally for exactly this reason. (The assert checks the physics: every branch carries the same 1 A through 1 Ω, so the far node sits at 3,000 V above the reference.)

### Why power flow is nonlinear

Take the simplest grid: a source at fixed voltage **Vs** feeds a load through a line with resistance **R**. The load does not draw a fixed current — it draws a fixed **power P** (a heater, a motor, a whole town).

With load voltage **V**, the line current is I = (Vs − V)/R, and the power arriving at the load is V · I. Requiring that to equal P:

**V · (Vs − V) / R = P**

The unknown V appears squared: the equation is **nonlinear**. Rearranged, V² − Vs·V + R·P = 0, a quadratic with two solutions:

**V = (Vs ± √(Vs² − 4RP)) / 2**

With Vs = 100 V, R = 1 Ω and P = 1,000 W:

In [5]:
Vs, R, P = 100.0, 1.0, 1000.0
disc = Vs**2 - 4 * R * P
V_high, V_low = (Vs + np.sqrt(disc)) / 2, (Vs - np.sqrt(disc)) / 2
for V in (V_high, V_low):
    I_line = (Vs - V) / R
    print(f"V = {V:6.2f} V, I = {I_line:5.2f} A, load gets {V * I_line:7.1f} W, line loses {I_line**2 * R:6.1f} W")
print(f"maximum transferable power: Vs^2 / (4R) = {Vs**2 / (4 * R):.0f} W")

V =  88.73 V, I = 11.27 A, load gets  1000.0 W, line loses  127.0 W
V =  11.27 V, I = 88.73 A, load gets  1000.0 W, line loses 7873.0 W
maximum transferable power: Vs^2 / (4R) = 2500 W


Both solutions deliver exactly 1,000 W, but they are very different operating points. The **high-voltage solution** (88.7 V, 11.3 A, 127 W of losses) is the normal one. The **low-voltage solution** (11.3 V, 88.7 A) delivers the same power by pushing a huge current, losing 7.9 kW in the line — physically possible, but not a state any operator wants.

If P exceeds **Vs²/(4R) = 2,500 W**, the square root becomes negative: there is *no* solution. The line simply cannot deliver that much power, however low the voltage falls. This is **voltage collapse**, the subject of voltage stability in Module 04. Your first nonlinear equation already contains it.

### Newton's method in one dimension

A quadratic has a formula, but the AC power flow equations do not. The general tool is **Newton's method**. To solve f(x) = 0, replace f near the current guess xₖ by its tangent line (first-order Taylor expansion) and jump to where the tangent crosses zero:

f(xₖ + Δx) ≈ f(xₖ) + f′(xₖ) · Δx = 0  →  **xₖ₊₁ = xₖ − f(xₖ) / f′(xₖ)**

For our load: f(V) = V(Vs − V)/R − P (power mismatch, W) and f′(V) = (Vs − 2V)/R. Start at V = Vs — in power systems this "all voltages at nominal" guess is called a **flat start**.

In [6]:
f = lambda V: V * (Vs - V) / R - P          # power mismatch, W
df = lambda V: (Vs - 2 * V) / R             # derivative, W/V

V = Vs
rows = []
for k in range(6):
    rows.append({"iteration": k, "V": V, "mismatch_W": f(V), "error_V": abs(V - V_high)})
    V = V - f(V) / df(V)
print(pd.DataFrame(rows).to_string(index=False, formatters={"V": "{:.10f}".format,
      "mismatch_W": "{:.2e}".format, "error_V": "{:.2e}".format}))

 iteration              V mismatch_W  error_V
         0 100.0000000000  -1.00e+03 1.13e+01
         1  90.0000000000  -1.00e+02 1.27e+00
         2  88.7500000000  -1.56e+00 2.02e-02
         3  88.7298387097  -4.06e-04 5.25e-06
         4  88.7298334621  -2.75e-11 3.55e-13
         5  88.7298334621   0.00e+00 0.00e+00


Look at the error column: 11, 1.3, 0.02, 5·10⁻⁶, 4·10⁻¹³. Once Newton is close, the number of correct digits roughly **doubles every iteration** — **quadratic convergence**. That is why AC power flow solvers typically finish in 3–6 iterations even for huge networks.

But the starting point matters:

In [7]:
V = 10.0
for _ in range(20):
    V = V - f(V) / df(V)
print(f"start at 10 V -> converges to {V:.4f} V (the low-voltage solution)")
print(f"scipy brentq on [Vs/2, Vs]: {optimize.brentq(f, Vs / 2, Vs):.10f} V")

start at 10 V -> converges to 11.2702 V (the low-voltage solution)
scipy brentq on [Vs/2, Vs]: 88.7298334621 V


Started at 10 V, Newton finds the *other* root. It converges to whichever solution is nearby — it does not know which one is physically sensible. A flat start keeps you near the normal high-voltage solution. SciPy's `brentq` is a robust alternative in one dimension when you can bracket the root (here between Vs/2 and Vs), but it does not generalise to many unknowns. Newton does.

### Newton for a system: a small constant-power network

Now two loads. Source bus 0 is fixed at 100 V (the **slack**). Line 0–1 and line 1–2 have 1 Ω each. Bus 1 draws 800 W, bus 2 draws 600 W. Unknowns: V₁, V₂.

Power mismatch at each load bus (arriving power minus demand), with branch currents I₀₁ = V₀ − V₁ and I₁₂ = V₁ − V₂ (R = 1 Ω):

- f₁ = V₁ (I₀₁ − I₁₂) − 800 = V₁ (V₀ − 2V₁ + V₂) − 800
- f₂ = V₂ · I₁₂ − 600 = V₂ (V₁ − V₂) − 600

Newton needs the **Jacobian** J — the matrix of partial derivatives ∂fᵢ/∂Vⱼ — and solves a *linear* system in every iteration:

**J(Vₖ) · ΔV = −F(Vₖ),  Vₖ₊₁ = Vₖ + ΔV**

| | ∂/∂V₁ | ∂/∂V₂ |
|---|---|---|
| f₁ | V₀ − 4V₁ + V₂ | V₁ |
| f₂ | V₂ | V₁ − 2V₂ |

In [8]:
V0, P1, P2 = 100.0, 800.0, 600.0

def mismatch(v):
    v1, v2 = v
    return np.array([v1 * (V0 - 2 * v1 + v2) - P1, v2 * (v1 - v2) - P2])

def jacobian(v):
    v1, v2 = v
    return np.array([[V0 - 4 * v1 + v2, v1],
                     [v2, v1 - 2 * v2]])

v = np.array([V0, V0])                       # flat start
for k in range(6):
    print(f"iteration {k}: V = {v.round(6)}, max |mismatch| = {np.abs(mismatch(v)).max():.2e} W")
    v = v + np.linalg.solve(jacobian(v), -mismatch(v))

iteration 0: V = [100. 100.], max |mismatch| = 8.00e+02 W
iteration 1: V = [86. 80.], max |mismatch| = 1.20e+02 W
iteration 2: V = [82.459665 74.550989], max |mismatch| = 1.04e+01 W
iteration 3: V = [82.163903 74.064065], max |mismatch| = 9.31e-02 W
iteration 4: V = [82.161547 74.060008], max |mismatch| = 6.90e-06 W
iteration 5: V = [82.161547 74.060008], max |mismatch| = 2.27e-13 W


The same pattern: from the flat start the mismatch drops from 800 W to about 10⁻¹³ W in five iterations, with the correct digits roughly doubling each time. Every iteration is one **linear** solve — exactly the `np.linalg.solve` from EE-1.1, now applied to the Jacobian. AC power flow in Module 04 is this algorithm with complex voltages and many more buses.

Cross-check against SciPy, then validate physically: the source must supply the loads **plus** the line losses.

In [9]:
sol = optimize.root(mismatch, [V0, V0], jac=jacobian)
assert sol.success and np.allclose(sol.x, v)

v1, v2 = v
p_source = V0 * (V0 - v1)                    # W delivered by the slack
losses = (V0 - v1) ** 2 + (v1 - v2) ** 2      # I^2 R in both lines, W
assert np.isclose(p_source, P1 + P2 + losses), "power balance violated"
print(f"V1 = {v1:.3f} V, V2 = {v2:.3f} V")
print(f"source {p_source:.1f} W = loads {P1 + P2:.0f} W + losses {losses:.1f} W")

V1 = 82.162 V, V2 = 74.060 V
source 1783.8 W = loads 1400 W + losses 383.8 W


The slack supplies 1,783.8 W: 1,400 W for the loads and 383.8 W lost in the lines. Notice what the slack bus does: it is the only bus whose output is **not** fixed in advance — it covers whatever the losses turn out to be. That is the second job of the slack bus, after being the voltage reference from EE-1.1.

## Check your understanding

**Vectorisation and sparsity**

1. What shape does `availability * p_nom_MW` have, and what would happen if `p_nom_MW` had shape (8760,) instead of (6,)?
2. Why does the chain network's **G** have about 3 non-zeros per row? What would that number be for a meshed grid where each bus has 3–4 neighbours?

**Nonlinearity and Newton**

1. Raise P in the single-line example to 2,400 W, then 2,600 W. Predict what Newton does in each case, then run it.
2. Near P = 2,500 W, what happens to f′(V) at the solution? Why does that make Newton struggle? (Hint: look at the Newton step formula.)

**Thinking ahead**

1. In the two-load network, which entries of the Jacobian would be zero in a large network, and why does that matter for speed?
2. The slack bus covered the losses. In the PyPSA model of M1.2, were losses modelled at all? What did that assume?